In [15]:
from modules import creation_dataframe,CreationClipDataset,HeadClassifierClipModel,Train,CreationProcessedDataset, ClipExtractor
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor,CLIPImageProcessor,CLIPTokenizerFast,CLIPModel
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from torch.nn.modules.loss import BCEWithLogitsLoss,CrossEntropyLoss
import torch
from sklearn.utils.class_weight import compute_class_weight
import numpy as np


In [16]:
#Creation of the dataframes
train_df=creation_dataframe("../data/train.jsonl")

val_df=creation_dataframe("../data/dev.jsonl")

In [17]:
#Creation of the first Clip Datasets to get the texts and images embeddings through the pretrained clip model
train_clip_dataset=CreationClipDataset(train_df)
val_clip_dataset=CreationClipDataset(val_df)

In [18]:
#Initialisation of Clip Processors
#text_processor=CLIPTokenizerFast.from_pretrained("openai/clip-vit-base-patch32")
#image_processor=CLIPImageProcessor.from_pretrained("openai/clip-vit-base-patch32")
processor=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [19]:
#Initialisation of the clip model, the device, and the batch size
clip_model=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
batch_size=32

In [20]:
#Creation of the fisrt Clip Dataloaders to get the texts and images embeddings through the pretrained clip model

def collate_fn(batch):
    images=[b["images"] for b in batch]
    texts=[b["texts"] for b in batch]
    labels=[b["labels"] for b in batch]
    inputs=processor(images,texts,return_tensors="pt", padding=True, max_length=77, truncation=True)
    inputs["labels"]=torch.tensor(labels,dtype=torch.float32)
    return inputs

train_clip_dataloader=DataLoader(train_clip_dataset,batch_size=batch_size,shuffle=True,collate_fn=collate_fn)
val_clip_dataloader=DataLoader(val_clip_dataset,batch_size=batch_size,shuffle=True,collate_fn=collate_fn)

In [21]:
#clip_extractor=ClipExtractor(clip_model,device)
#final_train_data,final_val_data=clip_extractor.get_embeddings(train_clip_dataloader,val_clip_dataloader,"./modules/clip_embeddings")

In [22]:
train_data=torch.load("./modules/clip_embeddings/train_clip_embeddings.pt")
val_data=torch.load("./modules/clip_embeddings/val_clip_embeddings.pt")

In [23]:
train_dataset=CreationProcessedDataset(train_data)
val_dataset=CreationProcessedDataset(val_data)
train_dataloader=DataLoader(train_dataset,batch_size=32,shuffle=True)
val_dataloader=DataLoader(val_dataset,batch_size=32,shuffle=True)

In [24]:
model=HeadClassifierClipModel(fc_layer_sizes=[512])

In [25]:
class_weight=compute_class_weight("balanced",classes=np.unique(train_df["label"]),y=train_df["label"].to_numpy())
class_weight=torch.tensor(class_weight,dtype=torch.float32)
print(class_weight)

tensor([0.7798, 1.3934])


In [26]:
n_epochs=10
n_steps=(train_dataset.__len__()//batch_size)*n_epochs
optimizer=AdamW(model.parameters(),lr=0.01,weight_decay=0.1)
scheduler=get_linear_schedule_with_warmup(optimizer,num_warmup_steps=0.1*n_steps,num_training_steps=n_steps)
loss_fn=CrossEntropyLoss(weight=class_weight)

In [27]:
trainer=Train(processor,model,loss_fn,optimizer,n_epochs,train_dataloader,val_dataloader,scheduler,device,batch_size, patience=50, min_improvement=0.05)

In [28]:
trainer.run_training()

2026-02-19 14:30:04.247 | INFO     | modules.Train:run_training:43 - Epoch 0 :
2026-02-19 14:30:05.688 | INFO     | modules.Train:run_training:100 - Epoch 0: Train Loss = 0.6175630460108134
2026-02-19 14:30:05.689 | INFO     | modules.Train:run_training:101 - Epoch 0: Train Accuracy = 0.6634117647058824
2026-02-19 14:30:05.690 | INFO     | modules.Train:run_training:102 - Epoch 0: Train F1 = 0.5843382246113613
2026-02-19 14:30:05.691 | INFO     | modules.Train:run_training:104 - Epoch 0: Validation Loss = 0.7238384112715721
2026-02-19 14:30:05.692 | INFO     | modules.Train:run_training:105 - Epoch 0: Validation Accuracy = 0.586
2026-02-19 14:30:05.693 | INFO     | modules.Train:run_training:106 - Epoch 0: Validation F1 = 0.4595300261096606
2026-02-19 14:30:05.693 | INFO     | modules.Train:run_training:43 - Epoch 1 :
2026-02-19 14:30:07.179 | INFO     | modules.Train:run_training:100 - Epoch 1: Train Loss = 0.5569461186353425
2026-02-19 14:30:07.180 | INFO     | modules.Train:run_trai